In [ ]:
# This notebook is primarily for replacing text in pre-made character pages and exporting to the wiki.

import os
import json
import math
import numpy as np

# Edit these paths if needed.
home_path = '/home/lynnen/Dropbox/Rivals 2 Work/Export Data/Rivals2/Content/Characters' # REPLACE THIS WITH THE PATH TO YOUR EXPORTED FMODEL JSONS.

# Run this cell for function loading. No need for editing.
import pandas as pd
char_data = dict()

char_stat_labels = ["StrongAttackType", "DacusSpeedMultiplier", "Weight", "FrictionGround", "FrictionAir", "DashFrames", "DashSpeed", "DashAcceleration", "RunSpeedMax", "RunTurnAcceleration", "RunTurnFrames", "WalkAccelerationMax", "WalkSpeedMax", "Gravity", "HitstunGravity", "FallSpeedMax", "FastFallSpeed", "AirAcceleration", "AirSpeedHorizontalMax", "JumpSpeedHorizontalMax", "FullHopSpeed", "ShortHopSpeed", "DoubleJumpSpeed", "DoubleJumpMaxHorizontalSpeed", "DoubleJumpSpeedDecay", "MaxDoubleJumps", "PlatdropLandingLockoutFrames", "AirDodgeSpeed", "AirDodgeFriction", "RollSpeed", "LedgeStandSpeed", "LedgeRollSpeed", "LedgeJumpMaxHorizontalAirSpeed", "GetupRollSpeed", "TechRollSpeed", "WallJumpSpeed"]

custom_key_find = ["Weight", "Gravity", "Full Hop Speed", "Short Hop Speed", "Double Jump Speed", "Jump Speed Horizontal Max", "Run Speed Max", "Dash Speed", "Run Turn Acceleration", "Air Speed Horizontal Max", "Air Acceleration", "Run Turn Frames"]


char_attribute_exceptions = []

def processed_char(chara):
    if chara == 'La Reina':
        return "LaReina"
    else:
        return chara

# This prints all the relevant character stats for a character.
def character_stats(chara):
    output = ["{{RoA2-Character", f"| chara = {chara}"]
    with open(os.path.join(home_path, processed_char(chara), 'CD_' + processed_char(chara) + '.json')) as f:
        charnum = json.load(f)[1]['Properties']

        for label in char_stat_labels:
            if label in charnum.keys():
                if (label == 'WallJumpSpeed'):
                    output.append(f"| {label}X = {charnum[label]['X']}")    
                    output.append(f"| {label}Y = {charnum[label]['Y']}")    
                else:
                    output.append(f"| {label} = {charnum[label]}")

        output.append('}}')
        if('CustomAttributes' in charnum.keys()):
            custom_output = [f"{{{{RoA2-Character-CustomAttributes|chara={chara}"]
            custom_attr = []
            for pair in charnum['CustomAttributes']:
                # print(pair['Key'].find("Airdodge Speed"))
                analog = "N/A"
                # Guess the attr_analogs value based on the key
                if(pair['Key'].find("Airdodge Speed") >= 0):
                    analog = "Air Dodge Speed"
                elif(pair['Key'].find("Ground Friction") >= 0):
                    analog = "FrictionGround"
                elif(pair['Key'].find("Fall Speed") >= 0):
                    analog = "FallSpeedMax"
                elif(pair['Key'] == 'Superjump Vertical Speed'):
                    analog = "FullHopSpeed"
                else:
                    for keyword in custom_key_find:
                        if pair['Key'].find(keyword) >= 0:
                            analog = keyword.replace(' ', '')

                custom_attr.append([pair['Key'],pair['Value'],analog])
            custom_attr = np.transpose(custom_attr)
            custom_output.append(f"| attr_keys = {';'.join(custom_attr[0])}")
            custom_output.append(f"| attr_vals = {';'.join(custom_attr[1])}")
            custom_output.append(f"| attr_analogs = {';'.join(custom_attr[2])}")
            custom_output.append("}}")
            output.append('\n'.join(custom_output))
        

    return '\n'.join(output)


In [3]:
# Example
print(character_stats('Absa'))

{{RoA2-Character
| chara = Absa
| DacusSpeedMultiplier = 1.2
| Weight = 80
| FrictionGround = 0.89
| FrictionAir = 0.11
| DashFrames = 8
| DashSpeed = 11.33
| DashAcceleration = 4.5
| RunSpeedMax = 17.7
| RunTurnAcceleration = 4.25
| RunTurnFrames = 20
| WalkAccelerationMax = 0.56
| WalkSpeedMax = 8.5
| Gravity = 0.83
| HitstunGravity = 0.95
| FallSpeedMax = 17.0
| FastFallSpeed = 23.0
| AirAcceleration = 0.75
| AirSpeedHorizontalMax = 11.75
| JumpSpeedHorizontalMax = 15.5
| FullHopSpeed = 22.5
| ShortHopSpeed = 14.75
| DoubleJumpSpeed = -2.8333
| DoubleJumpMaxHorizontalSpeed = 12.5
| PlatdropLandingLockoutFrames = 14
| AirDodgeSpeed = 28.5
| AirDodgeFriction = 1.9
| RollSpeed = 26.9
| LedgeStandSpeed = 16.0
| LedgeRollSpeed = 32.0
| LedgeJumpMaxHorizontalAirSpeed = 11.0
| GetupRollSpeed = 26.1
| TechRollSpeed = 28.0
| WallJumpSpeedX = 17.0
| WallJumpSpeedY = 19.85
}}
{{RoA2-Character-CustomAttributes|chara=Absa
| attr_keys = Djump Accel
| attr_vals = 3.54
| attr_analogs = N/A
}}


In [4]:
import itertools
def intervals(a):
    a = sorted(set(a))
    b = []
    for k, g in itertools.groupby(enumerate(a), 
        key=lambda t: t[1] - t[0]):
        g = list(g)
        b.append([g[0][1], g[-1][1]]) 
    c = []
    for pair in b:
        if pair[0] == pair[1]:
            c.append(f"{pair[0]}")
        else:
            c.append(f"{pair[0]}-{pair[1]}")
    return ', '.join(c)
    
def default_move(chara, attack, image_count=1, custom_captions="", hitName=""):

    with open(os.path.join(home_path, processed_char(chara), 'Attacks', f'ATT_{processed_char(chara)[0:3]}_{attack}.json')) as f:
        props = json.load(f)[0]['Properties']

        hitbox_dict = dict()
        for attr in props['HitboxAttributes']:
            hitbox_dict[attr['Name']] = attr['OnHitPropertiesName']

        move_length = 0
        hitbox_track = dict()
        total_active_frames = list()
        iasa = 0
        for h in props['HitboxOnHitProperties']:
            hitbox_track[h['Name']] = list()
        for window in props['Windows']:
            window_hits = dict()
            window_active = set()
            for hit in window['HitboxWindows']:
                hit_length = window['WindowLengthFrames'] - hit['StartFrame']
                if(hit['LengthInFrames'] != 0 and hit['LengthInFrames'] < hit_length):
                    hit_length = hit['LengthInFrames']
                if hitbox_dict[hit['HitboxName']] not in window_hits.keys():
                    window_hits[hitbox_dict[hit['HitboxName']]] = set()
                window_hits[hitbox_dict[hit['HitboxName']]].update(list(map(lambda x: x+1, range(hit['StartFrame'] + move_length,hit['StartFrame'] + hit_length + move_length))))
                window_active.update(list(map(lambda x: x+1, range(hit['StartFrame'] + move_length,hit['StartFrame'] + hit_length + move_length))))
            if window['IasaFrame'] != -1:
                iasa = move_length + window['IasaFrame'] + 1  
            move_length += window['WindowLengthFrames']
            for k, v in window_hits.items():
                hitbox_track[k].append(intervals(v))
            if len(window_active) > 0:
                total_active_frames.append(window_active)

        output = ['{{RoA2-MoveMode', f'| chara = {chara}', f'| attack = {attack}', '| image = ' + ' \\ '.join(list(map(lambda x: f'RoA2_{chara}_{attack}_{x}.png', range(image_count)))), f'| caption = {custom_captions}', f'| hitbox = {chara}-{attack}']
        output.append(f'| totalActive = {', '.join(list(map(intervals, total_active_frames)))}')
        output.append(f'| iasa = {iasa}')
        output.append(f'| totalDuration = {move_length}')
        if(attack in ['Nair','Fair','Bair','Uair','Dair']):
            output.append(f'| landingLag = {props['LandingLagFrames']}')
        output.append(f'| hitID = {';'.join(list(hitbox_track.keys()))}')
        output.append(f'| hitActive = {';'.join(map(lambda x: ', '.join(x), (hitbox_track.values())))}')
        if hitName != '':
            output.append(f'| hitName = {hitName}')
        output.append('}}')
        return '\n'.join(output)

In [6]:
# This cell prints for the Imported Data section of Rivals 2 pages.
import os

throw_keys = {
    "Damage":None, 
    "BaseKnockback":None,
    "KnockbackScaling":0.0,
    "KnockbackAngle":None,
    "HitstunMultiplier":1.0,
    "bTechable":True,
    "ForceTumble":True,
    "HitstunAnimationStateOverride":"None"
}

hit_keys = {
    "Damage":None,
    "BaseKnockback":None,
    "KnockbackScaling":0,
    "KnockbackAngle": None,
    "HitEffectName":None,
    "HitSoundName":None
}

article_keys = {
    "ArticleName":None,
    "CeilingCollisionResponse":None,
    "EcbRadius":None,
    "EcbType":None,
    "GotHitReaction":None,
    "GroundCollisionResponse":None,
    "HasHitReaction":None,
    "ParryReaction":None,
    "PlatformData":None,
    "ShouldGetOutOfGroundOnSpawn":None,
    "WallCollisionResponse":None,
    "bCanBeDetectedByOwner":False,
    "bCanBeHitByOwner":False,
    "bCanBeThrown":False,
    "bCanDetectOwner":False,
    "bCanHitOwner":False,
    "bInheritOwnerChargeValue":False,
    "bIsAttachedToOwner":False,
    "bIsProjectile":False,
    "bRotateWithVelocity":False
}

adv_hit_keys = {
    "SpecialEffect":"None",
    "HitpauseMultiplier":1,
    "ExtraHitpauseForOpponent":0,
    "HitpauseMovementType":"None",
    "HitpauseMovementOffsetFromHost":None,
    "HitpauseMovementStrength":0.5,
    "SDIMultiplier":1,
    "ASDIMultiplier":-1,
    "bCanReverse":True,
    "bForceFlinch":False,
    "GroundTechable":True,
    "bIgnoresWeight":False,
    "bAutoFloorhuggable":False,
    "ProjectileInteraction":"Default",
    "bForceKnockbackInKnockdown":False,
    "bPreserveFacing":False,
    "KnockbackAngleMode":"SpecifiedAngle",
    "HitstunMultiplier":1,
    "HitfallHitstunMultiplier":1,
    "ParryReaction":"Stun",
    "GrabPartnerInteraction":"None",
    "ExtraShieldStun":0,
    "ShieldDamageMultiplier":1,
    "ShieldPushbackMultiplier":1,
    "ShieldHitpauseMultiplier":1,
    "FullChargeKnockbackMultiplier":1.3,
    "FullChargeDamageMultiplier":1.6,
    "FinalBaseKnockback":0,
    "ForceTumble":False,
    "IgnoreKnockbackArmor":False,
    "PreventChaingrabsOnHit":False,
    "HitstunAnimationStateOverride":"None"
}

def imported_data(chara):
    hits = list()
    throws = list()
    articles = list()

    for root, dirs, files in os.walk(os.path.join(home_path, processed_char(chara), 'Attacks')):
        for file in files:
            move_id = file.split('_', maxsplit=2)[2].split('.')[0]
            move_data = dict()
            
            path = os.path.join(home_path, processed_char(chara), 'Attacks', file)
            if('ART_' in file):
                path = os.path.join(home_path, processed_char(chara), 'Attacks', 'Articles', file)
                with open(path) as f:
                    lexicon = dict()
                    props = json.load(f)[0]
                    if('Properties' not in props.keys()):
                        continue
                    props = props['Properties']
                    s = ['{{RoA2-ArticleData', f'| chara = {chara}', f'| moveID = {move_id}']
                    for k, v in article_keys.items():
                        if(k not in props.keys()):
                            continue
                        value = props[k]
                        if(k=='PlatformData'):
                            if 'Solid' in value.keys() and not value['Solid']:
                                s.append(f'| PlatformDataSolid = false')
                        else:
                            if type(value) == str:
                                value = value.split('::')[-1]
                            if value != v:
                                s.append(f'| {k} = {value}')
                    s.append('}}')
                    articles.append('\n'.join(s))
            
            with open(path) as f:
                lexicon = dict()
                props = json.load(f)[0]
                if('Properties' not in props.keys()):
                    continue
                props = props['Properties']
                if('ThrowData' in props.keys()):
                    # print(path)
                    props = props
                    if (props['ThrowData']['ThrowReleaseData'][0]['ThrowKnockbackData']['Damage'] != 0):
                        throw_count = 0
                        for throw in props['ThrowData']['ThrowReleaseData']:
                            s = ['{{RoA2-TrueThrowData', f'| chara = {chara}', f'| moveID = {move_id}', f'| throwNumber = Throw{throw_count}']
                            for k, v in throw_keys.items():
                                value = throw['ThrowKnockbackData'][k]
                                if type(value) == str:
                                    value = value.split('::')[-1]
                                if value != v:
                                    s.append(f'| {k} = {value}')
                            s.append(f'| ReleaseOffsetX = {throw['ReleaseOffset']['X']}')
                            s.append(f'| ReleaseOffsetY = {throw['ReleaseOffset']['Y']}')
                            s.append('}}')
                            throw_count += 1
                            throws.append('\n'.join(s))
                        
                validOnHitProperties = list()
                if('HitboxAttributes' in props.keys()):
                    for hit in props['HitboxAttributes']:
                        validOnHitProperties.append(hit['OnHitPropertiesName'])
                # print(props.keys())
                # print(f)
                if('HitboxOnHitProperties' in props.keys()):
                    for hitboxKey in props['HitboxOnHitProperties']:
                        if hitboxKey['Name'] not in validOnHitProperties:
                            continue
                        s = ['{{RoA2-HitData', f'| chara = {chara}', f'| moveID = {move_id}', f'| nameID = {hitboxKey['Name']}']
                        for hitLabelKey, hitLabelDefault in hit_keys.items():
                            value = hitboxKey[hitLabelKey]
                            if type(value) == str:
                                value = value.split('::')[-1]
                            if value != hitLabelDefault:
                                s.append(f'| {hitLabelKey} = {value}')
                        for hitLabelKey, hitLabelDefault in adv_hit_keys.items():
                            value = hitboxKey['AdvancedOnHitProperties'][hitLabelKey]
                            if(hitLabelKey == 'HitpauseMovementOffsetFromHost'):
                                if(value['X'] != 0):
                                    s.append(f'| HitpauseMovementOffsetFromHostX = {value['X']}')
                                if(value['Y'] != 0):
                                    s.append(f'| HitpauseMovementOffsetFromHostY = {value['Y']}')
                            else:
                                if type(value) == str:
                                    value = value.split('::')[-1]
                                if value != hitLabelDefault:
                                    s.append(f'| {hitLabelKey} = {value}')
                        s.append('}}')
                        hits.append('\n'.join(s))
    return f"==Throw Data==\n{'\n\n'.join(throws)}\n==Article Data==\n{'\n\n'.join(articles)}\n==Hit Data==\n{'\n\n'.join(hits)}"

# Change this argument for different Rivals 2 characters.
print(imported_data('Fleet'))

==Throw Data==
{{RoA2-TrueThrowData
| chara = Fleet
| moveID = Bthrow
| throwNumber = Throw0
| Damage = 2
| BaseKnockback = 12.0
| KnockbackAngle = 135
| ReleaseOffsetX = -90.0
| ReleaseOffsetY = 110.0
}}

{{RoA2-TrueThrowData
| chara = Fleet
| moveID = Uthrow
| throwNumber = Throw0
| Damage = 3
| BaseKnockback = 11.0
| KnockbackAngle = 90
| ReleaseOffsetX = 10.0
| ReleaseOffsetY = 200.0
}}
==Article Data==
{{RoA2-ArticleData
| chara = Fleet
| moveID = Arrow
| ArticleName = Arrow
| CeilingCollisionResponse = Custom
| EcbRadius = 10.0
| EcbType = Simple
| GotHitReaction = Destroy
| GroundCollisionResponse = Custom
| HasHitReaction = Destroy
| ParryReaction = Reflect
| ShouldGetOutOfGroundOnSpawn = False
| WallCollisionResponse = Custom
| bCanDetectOwner = True
| bIsProjectile = True
| bRotateWithVelocity = True
}}

{{RoA2-ArticleData
| chara = Fleet
| moveID = Apple
| ArticleName = Apple
| CeilingCollisionResponse = Stop
| EcbRadius = 20.0
| EcbType = Complex
| GroundCollisionResponse =

In [ ]:
# This file only works with pages that follow the RawDataPages format.
# This is work-in-progress.

import re
is_flag = False
for file in os.listdir('RawDataPages'):
    new_file = list()
    with open(os.path.join('RawDataPages', file)) as f:
        for line in f:
            m = re.search('%%(.+)%%', line)
            # for x in m:
            if m != None:
                found_footage = m.group(1)
                
                if found_footage.startswith('character_stats'):
                    new_file.append(character_stats(found_footage.split(',')[1]))
                elif found_footage.startswith('default_move'):
                    new_file.append(default_move(found_footage.split(',')[1], found_footage.split(',')[2]))
                elif found_footage.startswith('imported_data'):
                    new_file.append(imported_data(found_footage.split(',')[1]))
            else:
                new_file.append(line)
    with open(f"FreshPages/{file.split('.')[0]}_New.md", 'w') as f:
        f.write(''.join(new_file))